# Fine-tuning and resuming

Fine-tuning begins with existing network parameters and preserves the encoder,
head shapes, preprocessing and class taxonomy. This offline example creates a
small local checkpoint in memory to demonstrate loading, freezing, unfreezing and
resumption. **Its weights are random, not pretrained.**

For real pretrained parameters, use `CellPose.from_pretrained(local_path,
device=torch.device("cpu")).model` instead of the demonstration model below.
Build targets using that checkpoint's class mapping and normalization; do not
silently replace its heads. The training page explains real dataset preparation.


In [ ]:
import numpy as np
import torch
from cellseg_models_pytorch.models.cellpose.cellpose_unet import cellpose_nuclei
from cellseg_models_pytorch.transforms.functional.cellpose import gen_flow_maps

torch.manual_seed(0)
torch.set_num_threads(2)
device = torch.device("cpu")
rows, cols = np.mgrid[:64, :64]
instances = np.zeros((64, 64), dtype=np.int32)
instances[(rows - 24)**2 + (cols - 24)**2 < 8**2] = 1
instances[(rows - 44)**2 + (cols - 44)**2 < 6**2] = 2
# Synthetic RGB pixels are only a smoke fixture, not independent real training data.
image = np.repeat((instances > 0)[..., None], 3, axis=2).astype(np.float32)
images = torch.from_numpy(image.transpose(2, 0, 1)).unsqueeze(0).repeat(2, 1, 1, 1)
types = torch.from_numpy((instances > 0).astype(np.int64)).unsqueeze(0).repeat(2, 1, 1)
flows = torch.from_numpy(gen_flow_maps(instances)).unsqueeze(0).repeat(2, 1, 1, 1)
assert flows.shape == (2, 2, 64, 64) and torch.isfinite(flows).all()
model = cellpose_nuclei(2, enc_name="resnet18", enc_pretrain=False).to(device)

initial_state = {name: value.detach().clone() for name, value in model.state_dict().items()}
model.load_state_dict(initial_state, strict=True)
encoder = model[model.enc_name]
for parameter in encoder.parameters():
    parameter.requires_grad_(False)
encoder_before = {name: value.detach().clone() for name, value in encoder.state_dict().items()}
optimizer = torch.optim.Adam((p for p in model.parameters() if p.requires_grad), lr=1e-4)

head_parameter = next(p for p in model.parameters() if p.requires_grad)
head_before = head_parameter.detach().clone()


Freezing parameters alone does not freeze BatchNorm statistics. Keep the frozen
encoder in evaluation mode after putting the rest of the network into training mode.


In [ ]:
model.train()
encoder.eval()
optimizer.zero_grad(set_to_none=True)
prediction = model(images.to(device))["nuc"]
loss = torch.nn.functional.cross_entropy(prediction.type_map, types.to(device))
loss = loss + torch.nn.functional.mse_loss(prediction.aux_map, flows.to(device))
assert torch.isfinite(loss)
loss.backward()
gradients = [p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
assert gradients and all(torch.isfinite(gradient).all() for gradient in gradients)
optimizer.step()
print("Smoke training loss:", float(loss.detach()))

assert all(torch.equal(value, encoder.state_dict()[name]) for name, value in encoder_before.items())
assert not torch.equal(head_before, head_parameter.detach())


Unfreeze for a lower-learning-rate phase. Rebuilding the optimizer here deliberately
starts fresh optimizer state; this is distinct from resuming an interrupted run.


In [ ]:
for parameter in encoder.parameters():
    parameter.requires_grad_(True)
parameter = next(encoder.parameters())
before = parameter.detach().clone()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
model.train()
optimizer.zero_grad(set_to_none=True)
prediction = model(images.to(device))["nuc"]
loss = torch.nn.functional.cross_entropy(prediction.type_map, types.to(device))
loss = loss + torch.nn.functional.mse_loss(prediction.aux_map, flows.to(device))
assert torch.isfinite(loss)
loss.backward()
gradients = [p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
assert gradients and all(torch.isfinite(gradient).all() for gradient in gradients)
optimizer.step()
print("Smoke training loss:", float(loss.detach()))
assert not torch.equal(before, parameter.detach())


In [ ]:
from io import BytesIO

# Use a new file opened with "xb" instead of this in-memory buffer for a real run.
checkpoint = BytesIO()
torch.save({
    "model": model.state_dict(), "optimizer": optimizer.state_dict(),
    "epoch": 1, "torch_rng": torch.get_rng_state(),
    "encoder": "resnet18", "class_ids": [0, 1],
}, checkpoint)
checkpoint.seek(0)
state = torch.load(checkpoint, map_location=device, weights_only=True)
restored = cellpose_nuclei(2, enc_name=state["encoder"], enc_pretrain=False).to(device)
restored.load_state_dict(state["model"], strict=True)
resumed_optimizer = torch.optim.Adam(restored.parameters(), lr=1e-5)
resumed_optimizer.load_state_dict(state["optimizer"])
torch.set_rng_state(state["torch_rng"])
model.eval()
restored.eval()
with torch.inference_mode():
    expected = model(images)["nuc"]
    actual = restored(images)["nuc"]
torch.testing.assert_close(actual.type_map, expected.type_map, rtol=0, atol=0)
torch.testing.assert_close(actual.aux_map, expected.aux_map, rtol=0, atol=0)
assert resumed_optimizer.state_dict()["param_groups"] == optimizer.state_dict()["param_groups"]
print("Reloaded network and optimizer for epoch", state["epoch"])


This training-state bundle is different from the inference checkpoints accepted by
`from_pretrained`; reload it with the matching architecture and `load_state_dict`.
For real training also retain Python/NumPy/data-loader/CUDA RNG state, dataset split,
preprocessing and freeze phase, and scheduler/scaler state if used. Here only Torch
randomness is used. Checkpoint files are untrusted inputs: use restricted loading
and never load an unknown pickle with `weights_only=False`.
